# 05 — PyTorch MLP with Adam

**Competition:** A Cloned Airbnb Booking Prediction Competition — K353  
**Course:** COMP 468 — Abdullah Gül University

Course-allowed techniques used here:
- **Multi-layer Perceptron** built in PyTorch (Module 12).
- **Adam optimizer** (Module 13).
- **StandardScaler** for numerics (Module 6).
- **Custom `KFoldTargetEncoder`** reused from notebook 04 (Module 7 pattern) for high-cardinality categoricals.
- **K-Fold cross-validation** with the same seed as notebooks 03/04 so OOFs blend cleanly (Module 8).

Outputs:
- `outputs/oof_MLP.npy`
- `outputs/submission_mlp.csv`

---

### Outline
1. Setup
2. Load features + redefine the KFoldTargetEncoder (self-contained)
3. Build preprocessing `ColumnTransformer`
4. Define MLP architecture in PyTorch
5. Training utility (one fold)
6. 5-fold CV with OOF collection
7. Refit on full data → Kaggle submission

## 1. Setup

In [1]:
import warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
N_SPLITS = 5
torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)

OUT_DIR = Path('../outputs'); OUT_DIR.mkdir(exist_ok=True, parents=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} on {device}')

PyTorch 2.12.0+cpu on cpu


## 2. Load Features

In [2]:
train_df = pd.read_parquet(OUT_DIR / 'train_features.parquet')
test_df  = pd.read_parquet(OUT_DIR / 'test_features.parquet')
TARGET = 'NumReserveDays2016Q3'; ID_COL = 'PropertyID'
y = train_df[TARGET].astype(float).values
X = train_df.drop(columns=[TARGET, ID_COL])
X_test = test_df.drop(columns=[ID_COL]).reindex(columns=X.columns)
test_ids = test_df[ID_COL].values

cat_all = X.select_dtypes(exclude='number').columns.tolist()
cardinality = {c: X[c].nunique(dropna=False) for c in cat_all}
cat_high = [c for c, n in cardinality.items() if n > 15]
cat_low  = [c for c, n in cardinality.items() if n <= 15]
num_cols = X.select_dtypes(include='number').columns.tolist()
print(f'num={len(num_cols)}  cat_low={cat_low}  cat_high={cat_high}')

num=130  cat_low=['ListingType', 'MetropolitanStatisticalArea', 'CancellationPolicy']  cat_high=['PropertyType', 'Neighborhood']


### KFoldTargetEncoder (Module-7 style custom transformer, repeated here for self-containment)

In [3]:
class KFoldTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols, n_splits=5, smoothing=20.0, random_state=42):
        self.cols = cols; self.n_splits = n_splits
        self.smoothing = smoothing; self.random_state = random_state

    def _smoothed_map(self, X_col, y):
        stats = pd.DataFrame({'cat': X_col, 'y': y}).groupby('cat')['y'].agg(['mean', 'count'])
        return ((stats['count'] * stats['mean'] + self.smoothing * self.global_mean_)
                / (stats['count'] + self.smoothing)).to_dict()

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        self.global_mean_ = float(y.mean())
        self.maps_ = {c: self._smoothed_map(X[c].astype(str).fillna('__nan__'), y)
                      for c in self.cols}
        return self

    def transform(self, X):
        Xo = X.copy()
        for c in self.cols:
            Xo[c] = (X[c].astype(str).fillna('__nan__').map(self.maps_[c])
                       .fillna(self.global_mean_).astype('float32'))
        return Xo

    def fit_transform(self, X, y=None, **kw):
        y = np.asarray(y, dtype=float); self.global_mean_ = float(y.mean())
        Xo = X.copy()
        for c in self.cols: Xo[c] = np.full(len(X), self.global_mean_, dtype='float32')
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        for tr, va in kf.split(X):
            for c in self.cols:
                m = self._smoothed_map(X[c].astype(str).fillna('__nan__').iloc[tr], y[tr])
                mapped = X[c].astype(str).fillna('__nan__').iloc[va].map(m)
                Xo.iloc[va, Xo.columns.get_loc(c)] = mapped.fillna(self.global_mean_).astype('float32').values
        self.maps_ = {c: self._smoothed_map(X[c].astype(str).fillna('__nan__'), y) for c in self.cols}
        return Xo

    def get_feature_names_out(self, input_features=None):
        return np.array(input_features if input_features is not None else self.cols)
print('KFoldTargetEncoder ready.')

KFoldTargetEncoder ready.


## 3. Preprocessing `ColumnTransformer`
MLPs require **scaled** numeric inputs — that's the key difference vs. tree-based pipelines.

In [ ]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler',  StandardScaler()),
])
cat_low_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
cat_high_pipe = Pipeline([
    ('te', KFoldTargetEncoder(cols=cat_high, n_splits=5, smoothing=20, random_state=RANDOM_STATE)),
    ('scaler', StandardScaler()),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipe, num_cols),
    ('cat_low', cat_low_pipe, cat_low),
    ('cat_high', cat_high_pipe, cat_high),
])
print('Preprocessor ready.')

Preprocessor ready.


## 4. MLP Architecture
Module 12 covers MLPs with Linear → activation → (BatchNorm/Dropout). Module 13 covers Adam.

We use:
- 3 hidden layers (256 → 128 → 64) — a moderate-sized network for ~120-200 features.
- ReLU activations + BatchNorm for stable training + Dropout for regularisation.
- Output layer: linear (regression).
- Loss: MSE.

In [5]:
class MLPRegressor(nn.Module):
    def __init__(self, in_dim, hidden=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

## 5. One-Fold Training Utility
Standard training loop with early stopping on validation MSE.

In [6]:
def train_one_fold(X_tr, y_tr, X_va, y_va, *,
                   max_epochs=60, batch_size=512, lr=1e-3, weight_decay=1e-5,
                   patience=8, verbose=False):
    X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
    y_tr_t = torch.tensor(np.log1p(y_tr), dtype=torch.float32)   # log1p target
    X_va_t = torch.tensor(X_va, dtype=torch.float32).to(device)
    y_va_arr = np.asarray(y_va, dtype=float)

    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    model = MLPRegressor(in_dim=X_tr.shape[1]).to(device)
    optim = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_mse = float('inf'); best_state = None; bad = 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in dl:
            xb = xb.to(device); yb = yb.to(device)
            optim.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optim.step()

        model.eval()
        with torch.no_grad():
            pred_va = model(X_va_t).cpu().numpy()
        pred_va = np.clip(np.expm1(pred_va), 0, 92)            # invert log1p
        va_mse = mean_squared_error(y_va_arr, pred_va)

        if va_mse < best_mse - 1e-4:
            best_mse = va_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
        if verbose:
            print(f'    epoch {epoch:3d} | val MSE = {va_mse:.3f} | best = {best_mse:.3f}')
        if bad >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        final_va = np.clip(np.expm1(model(X_va_t).cpu().numpy()), 0, 92)
    return model, best_mse, final_va

## 6. 5-Fold CV (consistent split with notebooks 03 / 04)

In [7]:
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof = np.zeros(len(y))
fold_mse = []
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    print(f'\n--- Fold {fold+1}/{N_SPLITS} ---')
    # Fit preprocessor on training fold, transform both
    pp = ColumnTransformer(transformers=[
        ('num', num_pipe, num_cols),
        ('cat_low', cat_low_pipe, cat_low),
        ('cat_high', cat_high_pipe, cat_high),
    ])
    X_tr_proc = pp.fit_transform(X.iloc[tr_idx], y[tr_idx]).astype('float32')
    X_va_proc = pp.transform(X.iloc[va_idx]).astype('float32')
    print(f'  features after preprocessing: {X_tr_proc.shape[1]}')

    model, va_mse, va_pred = train_one_fold(
        X_tr_proc, y[tr_idx], X_va_proc, y[va_idx], verbose=False,
    )
    oof[va_idx] = va_pred
    fold_mse.append(va_mse)
    print(f'  fold MSE = {va_mse:.3f}')

print(f'\nMLP | CV MSE = {np.mean(fold_mse):.3f} ± {np.std(fold_mse):.3f} | {(time.time()-t0)/60:.1f} min')
np.save(OUT_DIR / 'oof_MLP.npy', oof)


--- Fold 1/5 ---


  features after preprocessing: 143


  fold MSE = 209.807

--- Fold 2/5 ---


  features after preprocessing: 142


  fold MSE = 229.214

--- Fold 3/5 ---


...

  fold MSE = 218.789

--- Fold 4/5 ---


  features after preprocessing: 142


  fold MSE = 203.443

--- Fold 5/5 ---


  features after preprocessing: 142


  fold MSE = 217.001

MLP | CV MSE = 215.651 ± 8.704 | 1.1 min


## 7. Refit on Full Data → Kaggle Submission
Train an ensemble of 5 MLPs (one per fold, but on the full data with different seeds) and average their test predictions for stability.

In [8]:
pp_full = ColumnTransformer(transformers=[
    ('num', num_pipe, num_cols),
    ('cat_low', cat_low_pipe, cat_low),
    ('cat_high', cat_high_pipe, cat_high),
])
X_full_proc = pp_full.fit_transform(X, y).astype('float32')
X_test_proc = pp_full.transform(X_test).astype('float32')
print('features after preprocessing:', X_full_proc.shape[1])

# Use the last 10% as a tiny holdout for early stopping during the full-train fits
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.permutation(len(y))
split = int(0.9 * len(y))
tr_idx_full, va_idx_full = idx[:split], idx[split:]

preds = []
for seed in range(5):
    torch.manual_seed(RANDOM_STATE + seed)
    model, _, _ = train_one_fold(
        X_full_proc[tr_idx_full], y[tr_idx_full],
        X_full_proc[va_idx_full], y[va_idx_full],
        verbose=False,
    )
    model.eval()
    with torch.no_grad():
        p = model(torch.tensor(X_test_proc, dtype=torch.float32).to(device)).cpu().numpy()
    preds.append(np.clip(np.expm1(p), 0, 92))
    print(f'  seed {seed}: done')

test_pred = np.mean(preds, axis=0)
sub = pd.DataFrame({'Property[Q3]_test': test_ids, 'Pred': test_pred})
sub.to_csv(OUT_DIR / 'submission_mlp.csv', index=False)
print(f'Saved submission_mlp.csv | mean={test_pred.mean():.2f} min={test_pred.min():.2f} max={test_pred.max():.2f}')
sub.head()

features after preprocessing: 143


  seed 0: done


  seed 1: done


  seed 2: done


  seed 3: done


  seed 4: done
Saved submission_mlp.csv | mean=14.72 min=0.00 max=92.00


,Property[Q3]_test,Pred
0,795,0.103265
1,2515,61.947701
2,2595,1.753683
3,5099,26.899090
4,5107,26.516668


## Summary
- 3-layer PyTorch MLP (256→128→64) trained with **Adam** on `log1p(y)`.
- StandardScaler + KFoldTargetEncoder preprocessing — all leakage-safe.
- Early stopping on validation MSE keeps training cheap and prevents overfit.
- OOFs stored as `oof_MLP.npy` for the blender notebook.

### → Next: `06_Blend.ipynb`
Combine the OOFs from baselines + XGBoost + MLP with optimal weights, then ship the final Kaggle submission.